# Quick Start: Trace Data Access and Analysis

This quickstart walks through loading ASQ trace artifacts, exploring their schema, and running simple analyses.

In [6]:
DATA_DIR = "./data/quickstart_data_access/" # path to the data directory where the traces are stored

## Loading Entire Trace Collections

The first step is to build a `TraceCollectionConfig` that build the artifact paths given the configuration used for generating their traces (dataset, retriever, reranker, etc.). This ensures the computed paths point to the correct TSV files on disk.

In [7]:
from artifacts import TraceCollectionConfig

config = TraceCollectionConfig(
    dataset="hotpotqa-test",
    data_dir=DATA_DIR,
    retriever="BM25",
    retriever_k=1000,
    reranker="electra",
    reranker_k=3,
    agent_family="search_r1",
    agent_generator="Qwen-3B",
)
config

TraceCollectionConfig(dataset=hotpotqa-test, retriever=BM25, retriever_k=1000, reranker=electra, reranker_k=3, agent_family=search_r1, agent_generator=Qwen-3B)

Once the config is instantiated, we can instantiate the trace collection object (`TraceCollection`), which allows to load trace artifacts (answers, synthetic queries, etc.) from disk or from Hugging Face, and exposes them through artifact interface classes or easier analysis and inspection.

In [8]:
from artifacts import TraceCollection
trace = TraceCollection(config)

If traces need to be downloaded from our **Hugging Face dataset** ([fpezzuti/ASQ](https://huggingface.co/datasets/fpezzuti/ASQ)), the next step is to login. Else, if traces will be loaded from disk, the login can be skipped.

In [9]:
%pip install huggingface_hub
from huggingface_hub import login

# This will prompt you for the token directly in the output cell
login()

Note: you may need to restart the kernel to use updated packages.


Next, use `trace.load_data()` to load trace data into memory. This method firstly tries to load artifacts from local tsv files, if the local files are missing, it falls back to downloading them from our Hugging Face dataset ([fpezzuti/ASQ](https://huggingface.co/datasets/fpezzuti/ASQ)).

In [10]:
trace.load_data(load_docs=True, trim_outliers=False, filter_empty_answers=False, verbose=False)
print("-----------------------------------------------------------------")
print("Statistics about the loaded traces:")
print("Number of queries with an answer:", trace.answers.nunique_qids())
print("Number of queries with synthetic reformulations:", trace.iter_queries.nunique_qids())
print("Number of retrieved ranked lists:", len(trace.retrieved_docs.retrieved_docs_df))
print("-----------------------------------------------------------------")

Trying to read trace artifacts from local files.
-----------------------------------------------------------------
Statistics about the loaded traces:
Number of queries with an answer: 7405
Number of queries with synthetic reformulations: 228
Number of retrieved ranked lists: 888
-----------------------------------------------------------------


Details about `TraceCollection` usage and the other artifacts are given next.

# Loading Individual Artifacts

Artifacts can also loaded individually, without creating a full `TraceCollection`. This is useful when you only need a specific artifact type.

For simplicity, we start by calculate file paths using the default method `config.build_output_paths()`. However, different paths can be specified.

In [11]:
from trace_generation import OutputPaths

# compute the output paths for the artifacts based on the configuration
output_paths: OutputPaths = config.build_output_paths()

print("----------------------------------------------------- [Config] -----------------------------------------------------")
print(f"{config}\n")
print("--------------------------------------------------- [Config Dir] ---------------------------------------------------")
print(f"Artifacts are stored in directory: {output_paths.base_dir}\n")

print("------------------------------------------------- [Artifact paths] -------------------------------------------------")
print("Answers:", output_paths.answers)
print("Synthetic queries:", output_paths.iter_queries)
print("Ranked lists:", output_paths.retrieved_docs)
print("Thoughts:", output_paths.thoughts)
print("\n--------------------------------------------------------------------------------------------------------------------")

----------------------------------------------------- [Config] -----------------------------------------------------
TraceCollectionConfig(dataset=hotpotqa-test, retriever=BM25, retriever_k=1000, reranker=electra, reranker_k=3, agent_family=search_r1, agent_generator=Qwen-3B)

--------------------------------------------------- [Config Dir] ---------------------------------------------------
Artifacts are stored in directory: data/quickstart_data_access/traces/hotpotqa-test/BM25_k1000_electra_k3/search_r1/Qwen-3B

------------------------------------------------- [Artifact paths] -------------------------------------------------
Answers: data/quickstart_data_access/traces/hotpotqa-test/BM25_k1000_electra_k3/search_r1/Qwen-3B/answers.tsv
Synthetic queries: data/quickstart_data_access/traces/hotpotqa-test/BM25_k1000_electra_k3/search_r1/Qwen-3B/iter_queries.tsv
Ranked lists: data/quickstart_data_access/traces/hotpotqa-test/BM25_k1000_electra_k3/search_r1/Qwen-3B/retrieved_docs.tsv
Though

### Loading answers

We use the precomputed `output_paths.answers` here, but you can pass any valid path to an answers TSV (with `qid` and `answer` columns). This is useful if your artifacts live in a custom directory or you want to load a different run’s outputs.

The `artifacts.load_answers()` method reads the answers TSV from disk and returns an `AnswersArtifact` wrapper so you can query and manipulate the answers consistently. Details about this interface are given next.

In [12]:
from artifacts import load_answers

print("Loading answers from:", output_paths.answers)
answers = load_answers(output_paths.answers, filter_empty_answers=False, verbose=False) # load answers from tsv

print(type(answers))
answers.answers_df.head()

Loading answers from: data/quickstart_data_access/traces/hotpotqa-test/BM25_k1000_electra_k3/search_r1/Qwen-3B/answers.tsv
<class 'artifacts.answers.AnswersArtifact'>


,qid,answer
qid,,
5a8b57f25542995d1e6f1371,5a8b57f25542995d1e6f1371,Yes
5a8c7595554299585d9e36b6,5a8c7595554299585d9e36b6,Governor of Michigan
5a85ea095542994775f606a8,5a85ea095542994775f606a8,The Way of Kings
5adbf0a255429947ff17385a,5adbf0a255429947ff17385a,No
5a8e3ea95542995a26add48d,5a8e3ea95542995a26add48d,New York City


### Loading ranked lists

The ranked lists are stored in the `retrieved_docs.tsv` artifact, with columns `qid`, `iteration`, and `docid`. Use `load_retrieved_docs()` to load them into a `RetrievedDocsArtifact`, which lets you inspect iterations or per-query results.

In [13]:
from artifacts import load_retrieved_docs

print("Loading ranked lists from:", output_paths.retrieved_docs)
retrieved_docs = load_retrieved_docs(output_paths.retrieved_docs, first_it_1=True, verbose=False)
print(type(retrieved_docs))

retrieved_docs.retrieved_docs_df.head()

Loading ranked lists from: data/quickstart_data_access/traces/hotpotqa-test/BM25_k1000_electra_k3/search_r1/Qwen-3B/retrieved_docs.tsv
<class 'artifacts.retrieved_docs.RetrievedDocsArtifact'>


,qid,iteration,docid,rank
0,5a8361b65542992ef85e22a0,1,93469,1
1,5a8361b65542992ef85e22a0,1,50570990,2
2,5a8361b65542992ef85e22a0,1,93729,3
3,5ae5aba0554299546bf82f17,1,315567,1
4,5ae5aba0554299546bf82f17,1,55135020,2


### Loading Synthetic Queries

Synthetic queries are stored in `iter_queries.tsv` with columns `qid`, `iteration`, and `llm_query`. Load them with `load_iter_queries()` to get a `SyntheticQueriesArtifact`, then inspect iterations or per-query rewrites.

In [14]:
from artifacts import load_iter_queries

print("Loading synthetic queries from:", output_paths.iter_queries)
iter_queries = load_iter_queries(output_paths.iter_queries, first_it_1=True, verbose=False)
print(type(iter_queries))

iter_queries.iter_queries_df.head()

Loading synthetic queries from: data/quickstart_data_access/traces/hotpotqa-test/BM25_k1000_electra_k3/search_r1/Qwen-3B/iter_queries.tsv
<class 'artifacts.iter_queries.SyntheticQueriesArtifact'>


,qid,iteration,llm_query
qid,,,
5a8361b65542992ef85e22a0,5a8361b65542992ef85e22a0,1,What is the World Boxing Association (WBA)?
5ae5aba0554299546bf82f17,5ae5aba0554299546bf82f17,1,Which of Tara Strong major voice role in anima...
5ae37c765542992f92d822d4,5ae37c765542992f92d822d4,1,"What is ""Romeo + Juliet""?"
5ab69c25554299110f219a39,5ab69c25554299110f219a39,1,Which American audio engineer and clandestine ...
5a7a230e5542996a35c170ee,5a7a230e5542996a35c170ee,1,What year was the film that starred the compos...


## Analysing Traces
The `TraceCollection` interface exposes several methods that can be used to analyse its traces.

In [15]:
trace.helper()

----------------------------------------[ TraceCollection Helper ]-------------------------------------------------
The TraceCollection class exposes the following methods:

1. directly_answered_qids(): Return the set of qids that were directly answered (i.e., qids that are in the answers, but have no synthetic queries).
2. directly_failed_qids(): Return the set of qids that failed to be answered (i.e., qids that are not in the answers and have involved no search calls).
3. download_trace(): Download the trace artifacts from the Hugging Face repository with ID <repo_id> and load them into the TraceCollection instance.
4. failed_traces(): Return the set of qids that failed to be answered (i.e., qids that are not in the answers).
5. helper(): Print a helper message describing the TraceCollection class and its methods.
6. load_data(): Load the trace data (queries, answers, iterative queries, and optionally retrieved documents) from the output files based on the configuration.
7. original_

For exampe, we can find the set of qids of organic queries directly answered via parametric knowledge:

In [16]:
trace.directly_answered_qids()
print("Directly answered qids:", len(trace.directly_answered_qids()))

Directly answered qids: 7177


Else, the set of qids of organic queries with incomplete traces:

In [17]:
print("Failed traces:", len(trace.failed_traces()))

Failed traces: 0


Each `TraceCollection` object exposes the main artifact objects as fields:
- `answers` (`AnswersArtifact` object)
- `iter_queries` (`SyntheticQueriesArtifact` object)
- `retrieved_docs` (`RetrievedDocsArtifact` object)

In [18]:
print(type(trace.answers))
print(type(trace.iter_queries))
print(type(trace.retrieved_docs))

<class 'artifacts.answers.AnswersArtifact'>
<class 'artifacts.iter_queries.SyntheticQueriesArtifact'>
<class 'artifacts.retrieved_docs.RetrievedDocsArtifact'>


Each of these artifact classes provides helpful methods for data analysis and inspection after `trace.load_data()` populates them.  You can yous the artifacts' helpers to inspect available methods.

In [19]:
trace.answers.helper()

-------------------------------------[ AnswersArtifact Helper ]-------------------------------------
The AnswersArtifact class exposes the following methods:

1. exclude_qids(): Return a new AnswersArtifact that excludes the qids in the input iterable of qids.
2. get_answer_for_qid(): Returns the answer for the given qid.
3. helper()
4. num_answers(): Return the number of answers in the answers DataFrame.
5. nunique_qids(): Return the number of unique qids in the answers DataFrame.
6. subset_for_qids(): Return a new AnswersArtifact including only the provided qids.
7. unique_qids(): Return the list of unique qids in the answers DataFrame (as strings).

-------------------------------------------------------------------------------------------------------------


In [20]:
trace.retrieved_docs.helper()

-------------------------------------[ RetrievedDocsArtifact Helper ]-------------------------------------
The RetrievedDocsArtifact class exposes the following methods:

1. docid_sets_by_iteration(): Return a nested dictionary mapping iteration number to
2. docids_for_qid(): Return the list of docids retrieved for a specific qid.
3. format_runs_into_trec_format(): Format the retrieved documents into TREC run format and store in self.df_run.
4. get_it_run(): Return the subset of the formatted run DataFrame (self.df_run) corresponding to the specified iteration.
5. helper()
6. iteration_retrieved_docs(): Return a RetrievedDocsArtifact of retrieved documents for a specific iteration.
7. num_search_calls(): Return the number of search calls, which is the number of unique (qid, iteration) pairs in the retrieved_docs_df.
8. nunique_iterations(): Return the number of unique iterations in the retrieved_docs_df.
9. nunique_qids(): Return the number of unique qids in the retrieved_docs_df.
10. 

In [21]:
trace.iter_queries.helper()

-------------------------------------[ SyntheticQueriesArtifact Helper ]-------------------------------------
The SyntheticQueriesArtifact class exposes the following methods:

1. assign_subquery_ids(): Return a new SyntheticQueriesArtifact with an additional column 'subquery_id' that assigns a unique ID to each (qid, iteration) pair.
2. get_subqueries(): Return a dictionary mapping subquery_id to llm_query for the given qid. If qid is None, return the mapping for all qids.
3. helper()
4. include_original_queries(): Return a new SyntheticQueriesArtifact that includes the original queries from the input DataFrame.
5. it_queries(): Return a new SyntheticQueriesArtifact containing only the queries for the specified iteration.
6. iteration_counts(): Return a Series with the counts of queries per iteration.
7. max_iteration(): Maximum iteration number in the iter_queries_df.
8. mean_iteration(): Mean iteration number in the iter_queries_df.
9. mean_query_length(): Compute the mean query len

For example, for `qid="5a8361b65542992ef85e22a0"`:

In [22]:
qid = "5a8361b65542992ef85e22a0"
trace.iter_queries = trace.iter_queries.assign_subquery_ids() # assign subquery ids to the synthetic queries
print(f"Synthetic Queries: {trace.iter_queries.get_subqueries(qid)}")
print(f"Ranked Lists: {trace.retrieved_docs.docids_for_qid(qid)}")
print(f"Answer: {trace.answers.get_answer_for_qid(qid)}")

Synthetic Queries: {'5a8361b65542992ef85e22a0_iter=1': 'What is the World Boxing Association (WBA)?'}
Ranked Lists: ['93469', '50570990', '93729']
Answer: International Boxing Hall of Fame
